# RUL (Remaining Useful Life) Pipeline
# Step 1 — Load Stage 1's cleaned dataset, define RUL target

RUL = cycles remaining until a cell crosses the standard 80% SOH failure 
threshold, measured from a given point in its life. Unlike Stage 1's 
fixed-fraction SOH regression, RUL requires each cell to actually REACH 
80% SOH in the observed data -- cells that never cross it (censored data) 
need separate handling, not a fabricated label.

In [1]:
# Cell 1: Load Stage 1's cleaned dataset

import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/input/notebooks/kritikabenjwal/battery-data-unification-soh-pipeline/battery_data/unified_battery_dataset_cleaned.csv')
# adjust path once you see the exact folder name Kaggle assigns to the linked notebook's output

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nUnique cells:", df['cell_id'].nunique())
print("Sources:", df['source'].value_counts().to_dict())

print("\nSOH range check (confirm this is the cleaned version, not raw):")
print(df['soh'].describe())

Shape: (399750, 22)
Columns: ['cell_id', 'source', 'cathode', 'temperature', 'cycle_index', 'discharge_capacity_ah', 'charge_capacity_ah', 'discharge_energy_wh', 'charge_energy_wh', 'voltage_max', 'voltage_min', 'reference_capacity_ah', 'soh', 'soh_out_of_range_flag', 'is_restricted_soc', 'high_spike_flag', 'low_dropout_flag', 'is_rpt_row', 'is_failure_event', 'is_transient_dropout', 'is_low_confidence_cell', 'is_early_invalid']

Unique cells: 268
Sources: {'snl': 276496, 'mit': 114738, 'calce': 5766, 'nasa': 2750}

SOH range check (confirm this is the cleaned version, not raw):
count    398333.000000
mean          0.886723
std           0.256548
min           0.020088
25%           0.832619
50%           0.927340
75%           1.000009
max           4.981535
Name: soh, dtype: float64


### Step 2 — Identify which cells actually cross 80% SOH (eligible for RUL labeling)

Standard EOL threshold = 80% of reference capacity. Cells that never drop 
below 0.80 in the observed data are RIGHT-CENSORED — we don't know their 
true RUL, only that it's longer than what we observed. These need to be 
excluded from RUL regression labels (not fabricated), though they could 
still be used later for a classification-style "still healthy" signal.

Also exclude restricted-SOC cells (is_restricted_soc) and low-confidence 
cells (is_low_confidence_cell) from RUL entirely, same as Stage 1 — their 
capacity readings don't follow the same 0-100% reference framework.

In [2]:
# Cell 2: Identify cells that cross 80% SOH vs. censored (never reach it)

df_clean = df.dropna(subset=['soh']).copy()
df_clean = df_clean[~df_clean['is_restricted_soc'] & ~df_clean['is_low_confidence_cell']]

eol_threshold = 0.80

cell_eol_status = []
for cid, g in df_clean.groupby('cell_id'):
    g = g.sort_values('cycle_index')
    below_threshold = g[g['soh'] <= eol_threshold]
    reached_eol = len(below_threshold) > 0
    eol_cycle = below_threshold['cycle_index'].iloc[0] if reached_eol else np.nan
    max_cycle = g['cycle_index'].max()
    min_soh_observed = g['soh'].min()
    
    cell_eol_status.append({
        'cell_id': cid, 'source': g['source'].iloc[0], 'cathode': g['cathode'].iloc[0],
        'reached_eol': reached_eol, 'eol_cycle': eol_cycle, 
        'max_cycle_observed': max_cycle, 'min_soh_observed': min_soh_observed
    })

eol_df = pd.DataFrame(cell_eol_status)
print("Total eligible cells (post-exclusion):", len(eol_df))
print("\nCells that reached 80% SOH (labelable for RUL):", eol_df['reached_eol'].sum())
print("Cells censored (never reached 80%):", (~eol_df['reached_eol']).sum())

print("\nBy source, reached_eol breakdown:")
print(eol_df.groupby(['source', 'reached_eol']).size())

print("\nFor censored cells, how close did they get? (min_soh_observed distribution)")
print(eol_df[~eol_df['reached_eol']]['min_soh_observed'].describe())

Total eligible cells (post-exclusion): 242

Cells that reached 80% SOH (labelable for RUL): 133
Cells censored (never reached 80%): 109

By source, reached_eol breakdown:
source  reached_eol
calce   True            8
mit     False          97
        True           43
nasa    False           6
        True           27
snl     False           6
        True           55
dtype: int64

For censored cells, how close did they get? (min_soh_observed distribution)
count    109.000000
mean       0.835318
std        0.032638
min        0.800133
25%        0.817846
50%        0.824156
75%        0.834613
max        0.964824
Name: min_soh_observed, dtype: float64


### Step 3 — Decide RUL framing given 45% censoring rate (109/242)

133 labelable cells is a workable but reduced dataset. Options considered:
1. Train RUL regression ONLY on the 133 reached_eol=True cells (clean but 
   smaller, and heavily biased toward SNL/NASA which have low censoring)
2. Lower the threshold to 85% SOH (median censored cell got to 82.4%, so 
   this would reclaim a meaningful chunk without being a meaningless target)
3. Use survival-analysis style censored regression (more statistically 
   correct, but adds real complexity for a hackathon timeline)

Given the timeline, check option 2 first -- if it meaningfully increases 
labelable cells without moving the goalposts too far from "real" EOL, 
it's the pragmatic choice.

In [3]:
# Cell 3: Check impact of a slightly relaxed EOL threshold (85% instead of 80%)

def eol_status_at_threshold(df_clean, threshold):
    results = []
    for cid, g in df_clean.groupby('cell_id'):
        g = g.sort_values('cycle_index')
        below = g[g['soh'] <= threshold]
        reached = len(below) > 0
        results.append({'cell_id': cid, 'source': g['source'].iloc[0], 'reached': reached})
    return pd.DataFrame(results)

for thresh in [0.80, 0.82, 0.85, 0.88]:
    r = eol_status_at_threshold(df_clean, thresh)
    print(f"Threshold {thresh}: {r['reached'].sum()}/{len(r)} cells reach EOL "
          f"({r.groupby('source')['reached'].sum().to_dict()})")

Threshold 0.8: 133/242 cells reach EOL ({'calce': 8, 'mit': 43, 'nasa': 27, 'snl': 55})
Threshold 0.82: 166/242 cells reach EOL ({'calce': 8, 'mit': 71, 'nasa': 29, 'snl': 58})
Threshold 0.85: 222/242 cells reach EOL ({'calce': 8, 'mit': 124, 'nasa': 30, 'snl': 60})
Threshold 0.88: 230/242 cells reach EOL ({'calce': 8, 'mit': 130, 'nasa': 31, 'snl': 61})


### Step 4 — Lock in 85% SOH as the EOL threshold, compute RUL labels for all reaching cells

85% recovers 222/242 cells (92%) vs. 133/242 (55%) at the standard 80% -- 
big practical gain for a small definitional shift. State this choice 
explicitly in the write-up (industry standard is typically 80%; we use 85% 
here to maximize usable training data given dataset test-duration limits, 
and note this as a limitation/future-work item for judges).

RUL definition: for each observed cycle in a cell's life, RUL = 
(cycle at which SOH first crosses 85%) - (current cycle). Computed for 
every valid cycle in eligible cells, not just one point per cell -- this 
gives many more training examples per cell (RUL decreases as cycle 
increases, which is the actual learnable relationship).

In [4]:
# Cell 5: Compute RUL for every cycle in every EOL-reaching cell (85% threshold)

eol_threshold = 0.85

rul_rows = []
for cid, g in df_clean.groupby('cell_id'):
    g = g.sort_values('cycle_index')
    below = g[g['soh'] <= eol_threshold]
    if len(below) == 0:
        continue  # censored, skip
    eol_cycle = below['cycle_index'].iloc[0]
    
    # only use cycles BEFORE eol_cycle (after EOL, RUL is 0/negative, not meaningful to predict from)
    pre_eol = g[g['cycle_index'] < eol_cycle].copy()
    pre_eol['rul_cycles'] = eol_cycle - pre_eol['cycle_index']
    pre_eol['eol_cycle'] = eol_cycle
    rul_rows.append(pre_eol)

rul_df = pd.concat(rul_rows, ignore_index=True)
print("Total labeled rows:", len(rul_df))
print("Unique cells:", rul_df['cell_id'].nunique())
print("\nRUL distribution:")
print(rul_df['rul_cycles'].describe())
print("\nBy source:")
print(rul_df.groupby('source')['cell_id'].nunique())

Total labeled rows: 141296
Unique cells: 202

RUL distribution:
count    141296.000000
mean        640.686226
std         581.656978
min           1.000000
25%         207.000000
50%         462.000000
75%         873.000000
max        3181.000000
Name: rul_cycles, dtype: float64

By source:
source
calce      8
mit      124
nasa      10
snl       60
Name: cell_id, dtype: int64


In [5]:
# Cell 6: Find which NASA cells reached EOL but produced zero pre-EOL rows

nasa_reached = eol_df[(eol_df['source']=='nasa')]  # from Step 2's eol_df, at 80% threshold -- need to recompute at 0.85
nasa_at_85 = eol_status_at_threshold(df_clean[df_clean['source']=='nasa'], 0.85)
reached_cells_85 = set(nasa_at_85[nasa_at_85['reached']]['cell_id'])
present_in_rul = set(rul_df[rul_df['source']=='nasa']['cell_id'])

missing = reached_cells_85 - present_in_rul
print(f"NASA cells that reached 85% but have ZERO rows in rul_df: {len(missing)}")
print(missing)

# inspect one missing cell to confirm the "crosses threshold almost immediately" theory
if missing:
    example = list(missing)[0]
    g = df_clean[df_clean['cell_id']==example].sort_values('cycle_index')
    print(f"\n{example} full trajectory:")
    print(g[['cycle_index','discharge_capacity_ah','soh']].head(15).to_string())

NASA cells that reached 85% but have ZERO rows in rul_df: 20
{'NASA_B0031', 'NASA_B0051', 'NASA_B0052', 'NASA_B0029', 'NASA_B0049', 'NASA_B0045', 'NASA_B0041', 'NASA_B0036', 'NASA_B0047', 'NASA_B0034', 'NASA_B0053', 'NASA_B0056', 'NASA_B0044', 'NASA_B0038', 'NASA_B0055', 'NASA_B0048', 'NASA_B0033', 'NASA_B0054', 'NASA_B0040', 'NASA_B0030'}

NASA_B0031 full trajectory:
        cycle_index  discharge_capacity_ah       soh
397828            1               1.666675  0.833338
397829            2               1.832858  0.916429
397830            3               1.813984  0.906992
397831            4               1.804439  0.902219
397832            5               1.802148  0.901074
397833            6               1.771565  0.885782
397834            7               1.813993  0.906996
397835            8               1.801326  0.900663
397836            9               1.788372  0.894186
397837           10               1.776540  0.888270
397838           11               1.785621  0.

In [6]:
# Cell 7: Document the exclusion, confirm final eligible RUL dataset

print("Final RUL-eligible dataset:")
print(f"  Total labeled rows: {len(rul_df)}")
print(f"  Unique cells: {rul_df['cell_id'].nunique()}")
print(f"  By source: {rul_df.groupby('source')['cell_id'].nunique().to_dict()}")

print(f"\nExcluded (reached EOL threshold but with 0 pre-EOL cycles to learn from): "
      f"{len(missing)} NASA cells")
print(f"  These cells' first valid SOH reading was already <= 85%, leaving no fade trajectory.")

print("\nRUL distribution (final, sanity check):")
print(rul_df['rul_cycles'].describe())
print("\nRUL distribution by source:")
print(rul_df.groupby('source')['rul_cycles'].describe())

Final RUL-eligible dataset:
  Total labeled rows: 141296
  Unique cells: 202
  By source: {'calce': 8, 'mit': 124, 'nasa': 10, 'snl': 60}

Excluded (reached EOL threshold but with 0 pre-EOL cycles to learn from): 20 NASA cells
  These cells' first valid SOH reading was already <= 85%, leaving no fade trajectory.

RUL distribution (final, sanity check):
count    141296.000000
mean        640.686226
std         581.656978
min           1.000000
25%         207.000000
50%         462.000000
75%         873.000000
max        3181.000000
Name: rul_cycles, dtype: float64

RUL distribution by source:
          count         mean         std  min    25%    50%      75%     max
source                                                                      
calce     361.0    37.515235   33.406427  1.0   13.0   26.0    52.00   131.0
mit     91945.0   439.759954  328.051907  1.0  186.0  378.0   627.00  1873.0
nasa      274.0    24.036496   17.034724  1.0   10.0   21.0    36.75    65.0
snl     48716.

### Step 5 — Feature engineering for RUL: sliding-window health indicators per cycle

Unlike Stage 1 (one feature vector per cell, from a fixed early window), 
RUL needs a feature vector at EVERY labeled cycle, computed from that 
cycle's trailing history (e.g., the preceding 10-20 cycles) -- since RUL 
prediction is meant to work at any point in a cell's deployed life, not 
just at a fixed checkpoint.

Reuse Stage 1's health-indicator logic (fade slope, curvature, SOH level, 
std) but computed on a rolling trailing window ending at each labeled cycle, 
rather than a single fixed window per cell.

In [7]:
# Cell 8: Sliding-window feature engineering for RUL — test on ONE cell first before scaling to all 202

def compute_window_features(g, current_idx, window_size=15):
    """g: sorted df for one cell. current_idx: position of the row we're computing features FOR.
    Uses the `window_size` rows ending at (and including) current_idx."""
    start_idx = max(0, current_idx - window_size + 1)
    window = g.iloc[start_idx:current_idx+1]
    if len(window) < 3:
        return None
    
    cycles = window['cycle_index'].values
    soh_vals = window['soh'].values
    
    slope, _ = np.polyfit(cycles, soh_vals, 1)
    curvature = np.polyfit(cycles, soh_vals, 2)[0] if len(window) >= 5 else np.nan
    
    return {
        'soh_current': soh_vals[-1],
        'soh_window_mean': soh_vals.mean(),
        'soh_window_std': soh_vals.std(),
        'soh_fade_slope': slope,
        'soh_fade_curvature': curvature,
        'cycle_index': cycles[-1],
        'n_window_cycles': len(window),
    }

# TEST on one cell first
test_cell = rul_df[rul_df['source']=='snl']['cell_id'].iloc[0]
g_test = df_clean[df_clean['cell_id']==test_cell].sort_values('cycle_index').reset_index(drop=True)
print(f"Testing on {test_cell}, {len(g_test)} total cycles")

test_features = []
for idx in range(len(g_test)):
    feat = compute_window_features(g_test, idx)
    if feat is not None:
        test_features.append(feat)

test_features_df = pd.DataFrame(test_features)
print("\nGenerated features shape:", test_features_df.shape)
print(test_features_df.head(10).to_string())
print("\nFeature ranges (sanity check):")
print(test_features_df.describe())

Testing on SNL_18650_LFP_15C_0-100_0.5/1C_a, 4540 total cycles

Generated features shape: (4538, 7)
   soh_current  soh_window_mean  soh_window_std  soh_fade_slope  soh_fade_curvature  cycle_index  n_window_cycles
0     0.939255         0.938119        0.001209        0.001405                 NaN            3                3
1     0.938856         0.938303        0.001095        0.000783                 NaN            4                4
2     0.939397         0.938522        0.001072        0.000610           -0.000310            5                5
3     0.939422         0.938672        0.001035        0.000477           -0.000199            6                6
4     0.939358         0.938770        0.000988        0.000372           -0.000147            7                7
5     0.939511         0.938863        0.000956        0.000310           -0.000105            8                8
6     0.939491         0.938932        0.000922        0.000259           -0.000080            9      

In [8]:
# Cell 9: Spot-check on a fast-degrading NASA cell

test_cell2 = rul_df[rul_df['source']=='nasa']['cell_id'].iloc[0]
g_test2 = df_clean[df_clean['cell_id']==test_cell2].sort_values('cycle_index').reset_index(drop=True)
print(f"Testing on {test_cell2}, {len(g_test2)} total cycles")

test_features2 = []
for idx in range(len(g_test2)):
    feat = compute_window_features(g_test2, idx)
    if feat is not None:
        test_features2.append(feat)

test_features2_df = pd.DataFrame(test_features2)
print("\nGenerated features shape:", test_features2_df.shape)
print(test_features2_df.to_string())

Testing on NASA_B0005, 168 total cycles

Generated features shape: (166, 7)
     soh_current  soh_window_mean  soh_window_std  soh_fade_slope  soh_fade_curvature  cycle_index  n_window_cycles
0       0.917675         0.923027        0.004316       -0.005285                 NaN            3                3
1       0.917631         0.921678        0.004408       -0.003733                 NaN            4                4
2       0.917323         0.920807        0.004310       -0.002737            0.001071            5                5
3       0.917831         0.920311        0.004088       -0.001989            0.000869            6                6
4       0.917573         0.919920        0.003904       -0.001537            0.000638            7                7
5       0.912878         0.919040        0.004331       -0.001611            0.000282            8                8
6       0.912387         0.918301        0.004588       -0.001571            0.000172            9              

### Step 6 — Scale sliding-window feature engineering to all 202 RUL-eligible cells

Confirmed the window logic behaves sanely on both a slow-fade (SNL, 4500+ 
cycles) and fast-fade (NASA, 168 cycles) cell. Apply across the full 
rul_df cell set now.

In [9]:
# Cell 10: Full feature engineering across all 202 cells

all_features = []
eligible_cell_ids = rul_df['cell_id'].unique()

for cid in eligible_cell_ids:
    g = df_clean[df_clean['cell_id'] == cid].sort_values('cycle_index').reset_index(drop=True)
    for idx in range(len(g)):
        feat = compute_window_features(g, idx)
        if feat is not None:
            feat['cell_id'] = cid
            all_features.append(feat)

rul_features_df = pd.DataFrame(all_features)
print("Total feature rows generated:", len(rul_features_df))
print("Unique cells:", rul_features_df['cell_id'].nunique())

# merge with RUL labels on (cell_id, cycle_index)
rul_model_df = rul_features_df.merge(
    rul_df[['cell_id', 'cycle_index', 'rul_cycles', 'source', 'cathode', 'eol_cycle']],
    on=['cell_id', 'cycle_index'], how='inner'
)
print("\nAfter merging with RUL labels:", rul_model_df.shape)
print("Unique cells in final model dataset:", rul_model_df['cell_id'].nunique())

print("\nNull counts:")
print(rul_model_df.isna().sum())

print("\nSample:")
print(rul_model_df.head(5).to_string())

Total feature rows generated: 203723
Unique cells: 202

After merging with RUL labels: (140897, 12)
Unique cells in final model dataset: 196

Null counts:
soh_current             0
soh_window_mean         0
soh_window_std          0
soh_fade_slope          0
soh_fade_curvature    390
cycle_index             0
n_window_cycles         0
cell_id                 0
rul_cycles              0
source                  0
cathode               256
eol_cycle               0
dtype: int64

Sample:
   soh_current  soh_window_mean  soh_window_std  soh_fade_slope  soh_fade_curvature  cycle_index  n_window_cycles cell_id  rul_cycles source cathode  eol_cycle
0     0.872727         0.875758        0.004285       -0.004545                 NaN            3                3  CS2_21          60  calce     LCO         63
1     0.872727         0.875000        0.003936       -0.002727                 NaN            4                4  CS2_21          59  calce     LCO         63
2     0.872727         0.874545

In [10]:
# Cell 11: Find the 6 lost cells, check cathode nulls

rul_cells = set(rul_df['cell_id'].unique())
model_cells = set(rul_model_df['cell_id'].unique())
lost_cells = rul_cells - model_cells
print(f"Lost cells ({len(lost_cells)}):", lost_cells)

for cid in lost_cells:
    n_rows = len(rul_df[rul_df['cell_id']==cid])
    print(f"  {cid}: {n_rows} pre-EOL rows in rul_df")

# cathode nulls - which source?
print("\nCathode null rows by source:")
print(rul_model_df[rul_model_df['cathode'].isna()]['source'].value_counts())

Lost cells (6): {'NASA_B0043', 'CS2_38', 'NASA_B0046', 'SNL_18650_NMC_15C_0-100_0.5/1C_b', 'SNL_18650_NMC_35C_0-100_0.5/2C_a', 'SNL_18650_NMC_15C_0-100_0.5/2C_b'}
  NASA_B0043: 1 pre-EOL rows in rul_df
  CS2_38: 2 pre-EOL rows in rul_df
  NASA_B0046: 1 pre-EOL rows in rul_df
  SNL_18650_NMC_15C_0-100_0.5/1C_b: 1 pre-EOL rows in rul_df
  SNL_18650_NMC_35C_0-100_0.5/2C_a: 1 pre-EOL rows in rul_df
  SNL_18650_NMC_15C_0-100_0.5/2C_b: 1 pre-EOL rows in rul_df

Cathode null rows by source:
source
nasa    256
Name: count, dtype: int64


In [11]:
# Cell 12: Backfill NASA cathode, finalize dataset for modeling

rul_model_df.loc[(rul_model_df['source']=='nasa') & (rul_model_df['cathode'].isna()), 'cathode'] = 'LCO'

print("Remaining nulls:")
print(rul_model_df.isna().sum())

print("\nFinal dataset for RUL model:")
print(f"  Rows: {len(rul_model_df)}, Cells: {rul_model_df['cell_id'].nunique()}")
print(rul_model_df.groupby('source')['cell_id'].nunique())

rul_model_df.to_csv('/kaggle/working/rul_model_dataset.csv', index=False)
print("\nSaved rul_model_dataset.csv")

Remaining nulls:
soh_current             0
soh_window_mean         0
soh_window_std          0
soh_fade_slope          0
soh_fade_curvature    390
cycle_index             0
n_window_cycles         0
cell_id                 0
rul_cycles              0
source                  0
cathode                 0
eol_cycle               0
dtype: int64

Final dataset for RUL model:
  Rows: 140897, Cells: 196
source
calce      7
mit      124
nasa       8
snl       57
Name: cell_id, dtype: int64

Saved rul_model_dataset.csv


### Step 7 — Train RUL regression model, GroupKFold by cell_id from the start

Reuse Stage 1's lesson: single train/test split is unreliable for smaller 
sources (calce=7, nasa=8 cells). Use GroupKFold(5) grouped by cell_id 
directly. Also worth checking: RUL scale varies hugely by source (SNL up 
to 3181 cycles, NASA/CALCE under 130) -- raw MAE in cycles will be 
dominated by SNL's scale, so report both raw MAE and a normalized metric 
(% of that cell's own EOL cycle) for a fairer cross-source read.

In [12]:
# Cell 13 (REPLACEMENT): add missing imports, then run as before

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols_rul = ['soh_current', 'soh_window_mean', 'soh_window_std', 
                     'soh_fade_slope', 'soh_fade_curvature', 'n_window_cycles']

X_rul = rul_model_df[feature_cols_rul].fillna(-1)
y_rul = rul_model_df['rul_cycles']
groups_rul = rul_model_df['cell_id']

gkf = GroupKFold(n_splits=5)
all_preds_rul = np.zeros(len(rul_model_df))
fold_maes_rul = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_rul, y_rul, groups_rul)):
    m = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, min_samples_leaf=5)
    m.fit(X_rul.iloc[train_idx], y_rul.iloc[train_idx])
    fold_preds = m.predict(X_rul.iloc[test_idx])
    all_preds_rul[test_idx] = fold_preds
    fold_mae = mean_absolute_error(y_rul.iloc[test_idx], fold_preds)
    fold_maes_rul.append(fold_mae)
    print(f"Fold {fold}: MAE={fold_mae:.1f} cycles, n_test={len(test_idx)}")

overall_mae_rul = mean_absolute_error(y_rul, all_preds_rul)
overall_r2_rul = r2_score(y_rul, all_preds_rul)
print(f"\nOverall CV MAE: {overall_mae_rul:.1f} cycles")
print(f"Overall CV R²: {overall_r2_rul:.4f}")

rul_cv_results = rul_model_df[['cell_id','source','rul_cycles','eol_cycle']].copy()
rul_cv_results['predicted_rul'] = all_preds_rul
rul_cv_results['abs_error'] = (rul_cv_results['rul_cycles'] - rul_cv_results['predicted_rul']).abs()
rul_cv_results['normalized_error_pct'] = rul_cv_results['abs_error'] / rul_cv_results['eol_cycle'] * 100

print("\nPer-source: raw MAE (cycles) vs normalized error (% of cell's EOL cycle):")
print(rul_cv_results.groupby('source').agg(
    raw_mae=('abs_error','mean'),
    normalized_pct=('normalized_error_pct','mean'),
    count=('cell_id','count')
))

Fold 0: MAE=244.6 cycles, n_test=28180
Fold 1: MAE=220.5 cycles, n_test=28178
Fold 2: MAE=230.0 cycles, n_test=28181
Fold 3: MAE=196.2 cycles, n_test=28178
Fold 4: MAE=226.8 cycles, n_test=28180

Overall CV MAE: 223.6 cycles
Overall CV R²: 0.5982

Per-source: raw MAE (cycles) vs normalized error (% of cell's EOL cycle):
           raw_mae  normalized_pct  count
source                                   
calce   602.793567     1364.961447    345
mit     131.811684       15.362883  91697
nasa    548.516037     1455.138650    256
snl     392.481086       25.495502  48599


### Step 8 — CALCE/NASA normalized error >1000% is a metric artifact, not a real finding. Check eol_cycle scale before drawing conclusions.

CALCE and NASA have very short lifespans (eol_cycle likely 20-130 based on 
earlier RUL distribution). Dividing a raw error of even 100 cycles by a 
tiny eol_cycle like 20 produces a nonsensical 500% figure. Check the raw 
numbers directly, and consider whether raw MAE (in cycles) is actually the 
fairer comparison here, not the normalized version.

In [13]:
# Cell 14: Check actual scale mismatch driving the inflated normalized error

print("eol_cycle distribution by source:")
print(rul_cv_results.groupby('source')['eol_cycle'].describe())

print("\nExample: a few CALCE rows, raw vs normalized")
print(rul_cv_results[rul_cv_results['source']=='calce'][['rul_cycles','predicted_rul','abs_error','eol_cycle','normalized_error_pct']].head(10).to_string())

eol_cycle distribution by source:
          count         mean         std    min     25%     50%     75%  \
source                                                                    
calce     345.0    76.481159   44.218542   26.0    40.0    63.0   132.0   
mit     91697.0   880.193016  360.370535  133.0   622.0   826.0  1019.0   
nasa      256.0    50.402344   15.623958    6.0    40.0    54.0    60.0   
snl     48599.0  2058.922941  762.257067    4.0  1831.0  2322.0  2475.0   

           max  
source          
calce    132.0  
mit     1874.0  
nasa      66.0  
snl     3182.0  

Example: a few CALCE rows, raw vs normalized
   rul_cycles  predicted_rul   abs_error  eol_cycle  normalized_error_pct
0          60      40.621277   19.378723         63             30.759878
1          59      38.875137   20.124863         63             31.944227
2          58     636.439413  578.439413         63            918.157798
3          57     611.857483  554.857483         63            880.7261

### Step 9 — Real bug found: model predicts RUL values 10x longer than the cell's entire observed lifespan. Raw-cycle RUL regression needs a scale-aware fix, not just better features.

The model has learned long-lifespan patterns from SNL/MIT (dominant by row 
count: 91697+48599 of 140897 rows) and misapplies them to short-lived 
CALCE/NASA cells. Options: 
(1) predict RUL as a FRACTION of remaining life relative to a per-source 
    or per-cell scale, rather than raw cycles
(2) train separate models per source/chemistry cluster
(3) add total-cycles-so-far or a source/scale indicator as an explicit 
    feature so the model can condition on regime
Try (1) first -- reframe target as rul_fraction = rul_cycles / eol_cycle, 
predicting a 0-1 value instead of raw cycles. This is also a more honest 
metric for cross-source comparison since it's scale-free by construction.

In [14]:
# Cell 15: Reframe target as RUL fraction (0-1), re-train, re-evaluate

rul_model_df['rul_fraction'] = rul_model_df['rul_cycles'] / rul_model_df['eol_cycle']
print("rul_fraction distribution:")
print(rul_model_df['rul_fraction'].describe())

X_rul2 = rul_model_df[feature_cols_rul].fillna(-1)
y_rul_frac = rul_model_df['rul_fraction']
groups_rul2 = rul_model_df['cell_id']

all_preds_frac = np.zeros(len(rul_model_df))
fold_maes_frac = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_rul2, y_rul_frac, groups_rul2)):
    m = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, min_samples_leaf=5)
    m.fit(X_rul2.iloc[train_idx], y_rul_frac.iloc[train_idx])
    fold_preds = m.predict(X_rul2.iloc[test_idx])
    all_preds_frac[test_idx] = fold_preds
    fold_mae = mean_absolute_error(y_rul_frac.iloc[test_idx], fold_preds)
    fold_maes_frac.append(fold_mae)
    print(f"Fold {fold}: MAE={fold_mae:.4f} (fraction), n_test={len(test_idx)}")

overall_mae_frac = mean_absolute_error(y_rul_frac, all_preds_frac)
overall_r2_frac = r2_score(y_rul_frac, all_preds_frac)
print(f"\nOverall CV MAE (fraction): {overall_mae_frac:.4f}")
print(f"Overall CV R²: {overall_r2_frac:.4f}")

rul_cv_results2 = rul_model_df[['cell_id','source','rul_fraction','eol_cycle']].copy()
rul_cv_results2['predicted_fraction'] = all_preds_frac
rul_cv_results2['abs_error_fraction'] = (rul_cv_results2['rul_fraction'] - rul_cv_results2['predicted_fraction']).abs()

print("\nPer-source MAE in fraction terms (0-1 scale, directly comparable across sources):")
print(rul_cv_results2.groupby('source')['abs_error_fraction'].agg(['mean','max','count']))

# sanity check on the same CALCE cell that broke before
print("\nSame CALCE example, now with fraction-based predictions:")
check = rul_model_df[rul_model_df['cell_id']=='CS2_21'][['cycle_index','rul_cycles','eol_cycle','rul_fraction']].head(10)
check['predicted_fraction'] = all_preds_frac[rul_model_df['cell_id']=='CS2_21'][:10]
print(check.to_string())

rul_fraction distribution:
count    140897.000000
mean          0.498489
std           0.287442
min           0.000314
25%           0.249603
50%           0.498456
75%           0.747409
max           0.999057
Name: rul_fraction, dtype: float64
Fold 0: MAE=0.0961 (fraction), n_test=28180
Fold 1: MAE=0.1037 (fraction), n_test=28178
Fold 2: MAE=0.0953 (fraction), n_test=28181
Fold 3: MAE=0.0938 (fraction), n_test=28178
Fold 4: MAE=0.0957 (fraction), n_test=28180

Overall CV MAE (fraction): 0.0969
Overall CV R²: 0.7459

Per-source MAE in fraction terms (0-1 scale, directly comparable across sources):
            mean       max  count
source                           
calce   0.272828  0.850271    345
mit     0.066873  0.737618  91697
nasa    0.163676  0.595795    256
snl     0.151980  0.820040  48599

Same CALCE example, now with fraction-based predictions:
   cycle_index  rul_cycles  eol_cycle  rul_fraction  predicted_fraction
0            3          60         63      0.952381         

### Step 10 — Fraction-based RUL confirmed as the right framing. Save model, results, and Stage 2 summary.

Final numbers: CV MAE 0.097 (fraction of remaining life), R² 0.746. 
Per-source: MIT 0.067, SNL 0.152, NASA 0.164, CALCE 0.273 -- same 
sample-size-driven pattern as Stage 1, worth reporting transparently. 
This target (RUL as % of life remaining) is also more actionable for a 
fleet operator than raw cycles, since it translates directly to "this 
battery has ~X% of its useful life left" regardless of chemistry/duty cycle.

In [15]:
# Cell 16: Save RUL model, results, summary
import joblib
final_rul_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, min_samples_leaf=5)
final_rul_model.fit(X_rul2, y_rul_frac)
joblib.dump(final_rul_model, '/kaggle/working/rul_final_model.pkl')

rul_cv_results2.to_csv('/kaggle/working/rul_cv_results.csv', index=False)

rul_summary = {
    'model': 'RandomForestRegressor',
    'target': 'rul_fraction (remaining cycles / eol_cycle, 0-1 scale)',
    'eol_threshold_soh': 0.85,
    'validation_method': 'GroupKFold (5-fold, grouped by cell_id)',
    'cv_mae_fraction': overall_mae_frac,
    'cv_r2': overall_r2_frac,
    'n_cells_total': rul_model_df['cell_id'].nunique(),
    'n_rows_total': len(rul_model_df),
    'per_source_mae': rul_cv_results2.groupby('source')['abs_error_fraction'].mean().to_dict(),
    'note': 'Raw-cycle RUL regression was tried first and found to badly overpredict for '
            'short-lived cells (CALCE/NASA) due to contamination from long-lived SNL/MIT '
            'training examples -- fraction-of-life framing corrected this and is the '
            'reported approach.',
}

import json
with open('/kaggle/working/stage2_rul_summary.json', 'w') as f:
    json.dump(rul_summary, f, indent=2, default=str)

print("=== STAGE 2 (RUL) SUMMARY ===")
for k, v in rul_summary.items():
    print(f"  {k}: {v}")

=== STAGE 2 (RUL) SUMMARY ===
  model: RandomForestRegressor
  target: rul_fraction (remaining cycles / eol_cycle, 0-1 scale)
  eol_threshold_soh: 0.85
  validation_method: GroupKFold (5-fold, grouped by cell_id)
  cv_mae_fraction: 0.09690900564789601
  cv_r2: 0.7458564598249948
  n_cells_total: 196
  n_rows_total: 140897
  per_source_mae: {'calce': 0.2728279976253748, 'mit': 0.06687313202925228, 'nasa': 0.16367572972164177, 'snl': 0.15198040978406477}
  note: Raw-cycle RUL regression was tried first and found to badly overpredict for short-lived cells (CALCE/NASA) due to contamination from long-lived SNL/MIT training examples -- fraction-of-life framing corrected this and is the reported approach.
